# CM4IR baseline on Google Colab
Select **Runtime → Change runtime type → GPU**. This notebook runs the authors' LSUN Cat SR×4 example. Full paper reproduction requires the authors' validation dataset and all checkpoints; see `REPRODUCE.md`. Outputs go to `/content/CM4IR/exp/image_samples`.


In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing'
print('PyTorch', torch.__version__)


In [ ]:
%cd /content
!git clone https://github.com/tirer-lab/CM4IR.git
%cd /content/CM4IR
!git rev-parse HEAD
!pip install -q einops lpips tqdm pillow pyyaml pandas scipy
!python -m pip freeze > /content/cm4ir_pip_freeze.txt


In [ ]:
from pathlib import Path
import urllib.request
ckpt = Path('/content/CM4IR/exp/logs/lsun_cat/cd_cat256_lpips.pt')
ckpt.parent.mkdir(parents=True, exist_ok=True)
if not ckpt.exists():
    urllib.request.urlretrieve('https://openaipublic.blob.core.windows.net/consistency/cd_cat256_lpips.pt', ckpt)
print('checkpoint bytes:', ckpt.stat().st_size)


Download the **authors' LSUN Cat validation set** from the link in the upstream README. Put images in a class subfolder under `/content/CM4IR/exp/datasets/lsun_cat/` (ImageFolder requires this layout). For persistent storage, mount Drive and copy the dataset there first.


In [ ]:
!pip install -q gdown
!mkdir -p /content/CM4IR/exp/datasets/lsun_cat/cat
!gdown --folder --remaining-ok 'https://drive.google.com/drive/folders/1b69dtdPQYjuYnmADzkxSyhPNqjQb6tyN' -O /content/CM4IR/exp/datasets/lsun_cat/cat
from pathlib import Path
images = list(Path('/content/CM4IR/exp/datasets/lsun_cat/cat').glob('*.png'))
print('validation images:', len(images))
assert len(images) >= 1, 'Author-provided LSUN Cat images are missing'


In [ ]:
from pathlib import Path
p = Path('/content/CM4IR/guided_diffusion/unet.py')
s = p.read_text()
a = s.index('class QKVFlashAttention(nn.Module):')
b = s.index('def count_flops_attn', a)
replacement = '''class QKVFlashAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, **kwargs):
        super().__init__()
        self.num_heads = num_heads

    def forward(self, qkv, attn_mask=None, key_padding_mask=None, need_weights=False):
        from einops import rearrange
        if attn_mask is not None or key_padding_mask is not None or need_weights:
            raise NotImplementedError("SDPA fallback does not support attention masks or weights")
        packed = rearrange(qkv, "b (three h d) s -> b three h s d", three=3, h=self.num_heads)
        q, k, v = packed.unbind(dim=1)
        out = F.scaled_dot_product_attention(q, k, v, dropout_p=0.0)
        return rearrange(out, "b h s d -> b (h d) s")


'''
p.write_text(s[:a] + replacement + s[b:])
%cd /content/CM4IR
!python main.py --config lsun_cat_256.yml --path_y lsun_cat --deg sr_bicubic --deg_scale 4 --sigma_y 0.05 -i CM4IR_lsun_cat_sr4_subset --iN 250 --gamma 0.2 --model_ckpt lsun_cat/cd_cat256_lpips.pt --subset_start 0 --subset_end {len(images)}
!find exp/image_samples -maxdepth 2 -type f | head
